# 1장 웹부록 | 파이썬으로 만드는 에니그마

「영화로 보는 AI 리터러시」 1장 '생각하는 기계의 꿈: 컴퓨터와 암호'의 코랩 실습입니다.

로터 3개와 반사판을 가진 작은 에니그마를 코드로 돌려 봅니다. 같은 글자가 매번 다르게 바뀌는지, 어떤 글자도 자기 자신이 되지 않는지 확인합니다. 끝에는 크립을 써서 후보 자리를 지워 나가는 간단한 해독기도 만듭니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.

## 1단계. 에니그마 만들기

책 3-4절에서 본 세 부품을 코드로 만듭니다.

- **플러그보드**: 전선으로 두 글자씩 맞바꿉니다.
- **로터**: 안에 얽힌 전선이 글자를 다른 글자로 바꾸는 바퀴입니다. 실제 독일군 로터 I, II, III의 배선을 그대로 씁니다.
- **반사판**: 신호를 되돌려 로터를 한 번 더 거치게 합니다.

키를 누를 때마다 오른쪽 로터가 한 칸 돕니다. 자동차 주행계처럼, 한 바퀴를 돌면 옆 로터가 한 칸 돕니다.
(실제 기계의 '링 설정'과 '이중 회전' 같은 세부는 이해하기 쉽도록 뺐습니다.)

이 칸은 기계를 만들기만 합니다. 실행해도 아무것도 나오지 않는 것이 정상입니다.

In [ ]:
ABC = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# 실제 에니그마 I의 로터 배선 (A가 E로, B가 K로 ... 이어집니다)
ROTOR_WIRING = {
    "I":   "EKMFLGDQVZNTOWYHXUSPAIBRCJ",
    "II":  "AJDKSIRUXBLHWTMCQGZNPYFVOE",
    "III": "BDFHJLCPRTXVZNYEIWGAKMUSQO",
}
REFLECTOR_B = "YRUHQSLDPXNGOKMIEBFZCWVJAT"


class Enigma:
    def __init__(self, rotors=("I", "II", "III"), start="AAA", plugs=""):
        self.rotors = rotors                        # 왼쪽, 가운데, 오른쪽 로터
        self.pos = [ABC.index(c) for c in start]    # 로터 세 개의 시작 위치
        self.plug = {c: c for c in ABC}             # 플러그보드: 두 글자씩 맞바꾸기
        for pair in plugs.split():
            a, b = pair[0], pair[1]
            self.plug[a], self.plug[b] = b, a

    def step(self):
        """키를 누를 때마다 오른쪽 로터가 한 칸 돕니다. 주행계처럼 넘어가면 옆 로터도 돕니다."""
        self.pos[2] = (self.pos[2] + 1) % 26
        if self.pos[2] == 0:
            self.pos[1] = (self.pos[1] + 1) % 26
            if self.pos[1] == 0:
                self.pos[0] = (self.pos[0] + 1) % 26

    def _through_rotor(self, c, i, backward=False):
        wiring, p = ROTOR_WIRING[self.rotors[i]], self.pos[i]
        x = (ABC.index(c) + p) % 26
        y = wiring.index(ABC[x]) if backward else ABC.index(wiring[x])
        return ABC[(y - p) % 26]

    def press(self, c):
        """글자 하나를 눌렀을 때 켜지는 전구의 글자를 돌려줍니다."""
        self.step()                                  # 1) 먼저 로터가 돕니다
        c = self.plug[c]                             # 2) 플러그보드
        for i in (2, 1, 0):                          # 3) 오른쪽 → 왼쪽 로터
            c = self._through_rotor(c, i)
        c = REFLECTOR_B[ABC.index(c)]                # 4) 반사판에서 되돌아옵니다
        for i in (0, 1, 2):                          # 5) 왼쪽 → 오른쪽 로터
            c = self._through_rotor(c, i, backward=True)
        return self.plug[c]                          # 6) 다시 플러그보드

    def window(self):
        """기계 위 작은 창에 보이는 로터 세 개의 글자"""
        return "".join(ABC[p] for p in self.pos)

    def type(self, text):
        text = "".join(ch for ch in text.upper() if ch in ABC)   # 알파벳만 남깁니다
        return "".join(self.press(ch) for ch in text)

print("에니그마 준비 완료!")

## 2단계. 같은 글자를 계속 눌러 보기

A를 열 번 눌러 봅니다. 카이사르 암호라면 A는 늘 같은 글자가 됩니다. 에니그마는 어떨까요?
로터 창의 글자가 한 칸씩 바뀌는 것도 함께 보세요.

In [ ]:
SETTING = dict(rotors=("I", "II", "III"), start="AAA", plugs="AB CD EF")   # ← 시작 위치(start)를 바꿔 보세요

machine = Enigma(**SETTING)
for n in range(10):
    lamp = machine.press("A")
    print(f"{n+1:2d}번째로 A를 누름 → 로터 창 {machine.window()} → 전구: {lamp}")

같은 A인데 매번 다른 글자가 켜집니다. 그래서 '가장 많이 나온 글자는 E'라는 빈도 분석이 통하지 않습니다.

## 3단계. 암호문을 다시 치면 평문이 나온다

반사판 덕분에, **같은 설정**으로 맞춘 기계에 암호문을 치면 평문이 그대로 나옵니다. 보내는 기계와 받는 기계가 똑같아도 되는 이유입니다.

👉 **바꿔 볼 곳**: `MESSAGE`에 영어나 독일어 문장을 넣어 보세요. 알파벳만 쓰고 띄어쓰기는 자동으로 빠집니다.

In [ ]:
MESSAGE = "WETTERBERICHT HEUTE REGEN"   # ← 문장을 바꿔 보세요

sender = Enigma(**SETTING)
cipher = sender.type(MESSAGE)
print("평문  :", "".join(ch for ch in MESSAGE.upper() if ch in ABC))
print("암호문:", cipher)

receiver = Enigma(**SETTING)          # 받는 쪽도 같은 설정에서 시작합니다
print("해독  :", receiver.type(cipher))

wrong = Enigma(rotors=("I", "II", "III"), start="AAB", plugs="AB CD EF")   # 시작 위치가 한 칸만 달라도
print("설정이 한 칸 다를 때:", wrong.type(cipher))

## 4단계. 어떤 글자도 자기 자신이 되지 않는다

이번에는 무작위 글자 10만 개를 쳐 보고, 평문 글자와 암호문 글자가 같았던 횟수를 셉니다.
신호가 반사판에서 '갔던 길'로 되돌아오지 않기 때문에, 이 횟수는 늘 0이어야 합니다.
이 작은 성질이 에니그마의 발목을 잡았습니다.

In [ ]:
import random
rng = random.Random(0)
N = 100_000                                     # ← 쳐 볼 글자 수
plain = "".join(rng.choice(ABC) for _ in range(N))
cipher = Enigma(**SETTING).type(plain)
same = sum(1 for p, c in zip(plain, cipher) if p == c)
print(f"글자 {N:,}개 가운데 자기 자신으로 바뀐 글자: {same}개")

### 빈도 분석으로 비교해 보기

같은 영어 문장을 카이사르 암호(세 칸 밀기)와 에니그마로 바꾼 뒤, 글자마다 몇 번 나왔는지 막대그래프로 그립니다.
카이사르 암호는 원래 글의 '모양'이 그대로 옆으로 밀려 있을 뿐입니다. 에니그마는 모양이 평평하게 뭉개집니다.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

english = ("it was the best of times it was the worst of times it was the age of wisdom "
           "it was the age of foolishness it was the epoch of belief it was the epoch of incredulity "
           "it was the season of light it was the season of darkness it was the spring of hope "
           "it was the winter of despair we had everything before us we had nothing before us ") * 5
plain = "".join(ch for ch in english.upper() if ch in ABC)
caesar = "".join(ABC[(ABC.index(ch) + 3) % 26] for ch in plain)
enigma = Enigma(**SETTING).type(plain)

fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)
for ax, (name, txt) in zip(axes, [("Plain text", plain), ("Caesar (shift 3)", caesar), ("Enigma", enigma)]):
    cnt = Counter(txt)
    ax.bar(list(ABC), [cnt[c] for c in ABC], color="#5b8def")
    ax.set_title(name, loc="left")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## 5단계. 설정은 몇 가지나 될까 (그림 1-7)

책의 그림 1-7을 직접 계산합니다. 로터 5개 중 3개를 골라 순서대로 꽂고, 시작 위치를 정하고, 플러그보드에 전선 10개를 꽂는 경우의 수입니다.

In [ ]:
from math import factorial

rotor_choice = 5 * 4 * 3                     # 로터 5개 중 3개를 골라 순서대로: 60
start_pos = 26 ** 3                          # 로터 세 개의 시작 위치: 17,576
pairs = 10                                   # 플러그보드 전선 수
plugboard = factorial(26) // (factorial(26 - 2 * pairs) * factorial(pairs) * 2 ** pairs)
total = rotor_choice * start_pos * plugboard

print(f"로터 고르기·순서 : {rotor_choice:,}")
print(f"로터 시작 위치   : {start_pos:,}")
print(f"플러그보드 10쌍  : {plugboard:,}")
print(f"모두 곱하면      : {total:,} 가지")

per_second = 100_000_000                     # ← 1초에 확인하는 설정 수 (1억)
years = total / per_second / (60 * 60 * 24 * 365)
print(f"1초에 {per_second:,}개씩 확인하면 약 {years:,.0f}년이 걸립니다.")

그래서 블레츨리 파크는 다 해 보지 않고, **답이 될 수 없는 것을 빠르게 지웠습니다.**

## 6단계. 크립으로 후보 자리 지우기 (그림 1-8)

가로챈 암호문이 하나 있습니다. 이 안 어딘가에 '날씨'(WETTER)라는 단어가 들어 있다고 추측합니다. 이것이 **크립**입니다.
크립을 암호문 아래에 한 칸씩 밀며 대 봅니다. 위아래로 **같은 글자가 겹치는 자리**는 답이 될 수 없으니 지웁니다.

(이 암호문은 우리가 모르는 설정으로 만들어 두었습니다. 설정은 마지막 단계에서 찾습니다.)

In [ ]:
# 우리가 모르는 설정으로 만든 암호문 (가로챈 무선 신호라고 생각하세요)
_secret = Enigma(rotors=("I", "II", "III"), start="QED", plugs="")
INTERCEPT = _secret.type("ANXKOMMANDOWETTERVORHERSAGEFUERHEUTENACHTREGENUNDSTARKERWIND")
del _secret

CRIB = "WETTER"     # ← 다른 크립을 넣어 볼 수도 있습니다 (예: "REGEN")

print("맨 윗줄이 암호문입니다. 그 아래로 크립을 한 칸씩 밀며 대 봅니다.\n")
print(INTERCEPT)
survivors = []
for start in range(len(INTERCEPT) - len(CRIB) + 1):
    segment = INTERCEPT[start:start + len(CRIB)]
    clash = [i for i in range(len(CRIB)) if segment[i] == CRIB[i]]
    line = " " * start + CRIB
    if clash:
        mark = f"자리 {start:2d}: 지움 ({CRIB[clash[0]]} 아래 {segment[clash[0]]})"
    else:
        mark = f"자리 {start:2d}: 후보로 남김"
        survivors.append(start)
    print(f"{line:<{len(INTERCEPT)}}  {mark}")

total_pos = len(INTERCEPT) - len(CRIB) + 1
print(f"\n자리 {total_pos}곳 가운데 {total_pos - len(survivors)}곳을 지웠고, {len(survivors)}곳이 남았습니다: {survivors}")

## 7단계. 작은 봄브: 남은 자리마다 설정을 시험하기

이제 남은 자리마다 로터 시작 위치 17,576가지를 차례로 시험합니다. 그 자리에서 암호문을 풀었을 때 정말 WETTER가 나오는 설정만 남깁니다.

진짜 봄브는 로터 순서와 플러그보드까지 함께 따졌습니다. 여기서는 간단하게 **로터 순서(I, II, III)는 알고, 플러그보드는 쓰지 않았다**고 가정합니다.
그래도 크립이 없으면 어느 자리를 시험할지조차 모른다는 점을 느껴 보세요. 몇 초쯤 걸립니다.

In [ ]:
import time
from itertools import product

def try_setting(start_letters, crib_pos):
    """이 시작 위치에서 crib_pos 자리의 암호문을 풀면 크립이 나오는지 확인합니다."""
    m = Enigma(rotors=("I", "II", "III"), start=start_letters)
    for _ in range(crib_pos):          # 크립 앞의 글자 수만큼 로터를 미리 돌립니다
        m.step()
    for k, want in enumerate(CRIB):
        if m.press(INTERCEPT[crib_pos + k]) != want:
            return False               # 하나라도 다르면 바로 버립니다
    return True

t0 = time.time()
found = []
tested = 0
for pos in survivors:
    for letters in product(ABC, repeat=3):
        tested += 1
        s = "".join(letters)
        if try_setting(s, pos):
            found.append((pos, s))
print(f"시험한 경우: {tested:,}가지, 걸린 시간 {time.time() - t0:.1f}초")

for pos, s in found:
    print(f"\n크립 자리 {pos}, 시작 위치 {s} 에서 모순이 없습니다!")
    print("해독한 평문:", Enigma(rotors=("I", "II", "III"), start=s).type(INTERCEPT))

봄브처럼 '모순이 없는 설정'에 이르면 멈췄습니다. 남은 후보는 실제로 해독해 보고 확인합니다. 블레츨리 파크에서도 이 마지막 확인은 사람이 했습니다.

## 생각해 볼 것

1. 4단계에서 '자기 자신으로 바뀐 글자'는 0개였습니다. 이 성질이 없었다면 6단계의 후보 자리 지우기는 어떻게 되었을까요?
2. 크립 WETTER가 통한 것은 독일군이 매일 같은 형식으로 날씨 보고를 보냈기 때문입니다. 여러분의 비밀번호에 숨은 '크립'은 무엇일까요? (생일, 전화번호, 자주 쓰는 단어)
3. 5단계에서 다 해 보면 수만 년이 걸렸지만, 크립과 봄브는 며칠, 몇 시간으로 줄였습니다. '다 해 보지 말고 지워 나가기'는 오늘날 컴퓨터 계산에서 왜 여전히 중요할까요?